# RewindKV — Validação da v0.3 (Kaggle, 2 × T4)

Testa a v0.3 (detector estrito + âncora de dúvida) em **134 problemas novos** (MATH-500 nível 5, todos), que não foram olhados no desenho da v0.3. Hipóteses e análise estão pré-registradas em `docs/resultados-experimentos.md`.

**Configuração (uma vez só):**
1. No painel da direita, em *Session options*: **Accelerator = GPU T4 x2** e **Internet = On**.
2. Em *Add-ons → Secrets*: adicione `GITHUB_TOKEN` (o mesmo token do Colab) e marque a caixa para ligá-lo a este notebook.

**Para rodar:** *Save Version → Save & Run All (Commit)*. Roda sozinho (~6 a 7 h); pode fechar a aba. Os resultados ficam na aba **Output** da versão salva.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código do projeto
Clona a tag `validacao-v0.3`: o código fica congelado no estado do pré-registro.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "GioNicolas/REWIND-KV"
TAG = "validacao-v0.3"
# Fora de /kaggle/working: o .git/config guarda a URL do clone, e o que fica em /kaggle/working vira Output.
DEST = "/tmp/REWIND-KV"

if not os.path.isdir(DEST):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    r = subprocess.run(["git", "clone", "-q", "--branch", TAG, f"https://{token}@github.com/{REPO}.git", DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd {DEST}
!pip install -q -e ".[dev]"

In [ ]:
!python -m pytest -q

## 2. Baixa o modelo e os problemas
Uma vez só, antes de abrir os dois processos (para eles não baixarem a mesma coisa ao mesmo tempo).

In [ ]:
import sys
from huggingface_hub import snapshot_download

sys.path.insert(0, f"{DEST}/src")  # o pip install -e só vale para o kernel depois de reiniciar
from rewind_kv.experiment import DEFAULT_MODEL, load_problems

snapshot_download(DEFAULT_MODEL)
print(len(load_problems("math500", 5)), "problemas de nível 5 (esperado: 134)")

## 3. Validação: um processo por GPU
Cada GPU roda metade dos problemas. Em cada problema, o **baseline** roda primeiro; se ele não detectar nenhuma falha, o resultado é copiado para `rewind_kv` e `blind_rewind` (com a mesma semente, a geração seria idêntica), senão os dois modos rodam de verdade.

A cada 10 min aparece a última linha de cada GPU.

In [ ]:
import glob, time

OUT = "/kaggle/working/validacao_v03"
os.makedirs(OUT, exist_ok=True)
PARTS = [(0, 0, 67), (1, 67, 67)]  # (GPU, primeiro problema, quantos)

procs = []
for gpu, start, n in PARTS:
    log = open(f"{OUT}/log_gpu{gpu}.txt", "w")
    cmd = ["python", "-u", "-m", "rewind_kv.experiment", "--dataset", "math500", "--level", "5",
           "--start", str(start), "--n", str(n), "--max-new-tokens", "4096",
           "--modes", "baseline", "rewind_kv", "blind_rewind", "--skip-untriggered",
           "--rewind-scope", "branch", "--anchor-style", "doubt", "--strict-detector", "--out", OUT]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env))

while any(p.poll() is None for p in procs):
    time.sleep(600)
    for gpu, _, _ in PARTS:
        done = [l for l in open(f"{OUT}/log_gpu{gpu}.txt", encoding="utf-8", errors="replace") if l.startswith("[")]
        print(time.strftime("%H:%M"), f"GPU{gpu}:", done[-1].strip() if done else "carregando...", flush=True)

print("códigos de saída (0 = ok):", [p.returncode for p in procs])
for gpu, _, _ in PARTS:
    print(f"\n--- GPU{gpu}, fim do log:")
    print("".join(open(f"{OUT}/log_gpu{gpu}.txt", encoding="utf-8", errors="replace").readlines()[-8:]))

## 4. Análise pré-registrada
- **H1 (principal):** nos problemas em que o baseline detectou falha, `rewind_kv` acerta mais que `blind_rewind`.
- **H2:** nesses problemas, `rewind_kv` acerta mais que o `baseline`.

Teste do sinal exato, bilateral, nos pares em que só um dos dois modos acertou. p < 0,05 conta como significativo.

In [ ]:
import json, math
import pandas as pd

rows = [json.loads(l) for p in sorted(glob.glob(f"{OUT}/*.jsonl")) for l in open(p, encoding="utf-8")]
df = pd.DataFrame(rows).drop_duplicates(["problem_idx", "mode"], keep="last")
acc = df.pivot(index="problem_idx", columns="mode", values="correct").astype(bool)
base = df[df["mode"] == "baseline"].set_index("problem_idx")
triggered = base.index[base["detections"] >= 1]

print(f"{len(acc)} problemas; {len(triggered)} com detecção no baseline (só neles os modos podem diferir)")
print(f"registros copiados do baseline: {int(df['copied_from_baseline'].sum())}\n")
print("Acurácia, todos:          ", acc.mean().round(3).to_dict())
print("Acurácia, com detecção:   ", acc.loc[triggered].mean().round(3).to_dict(), "\n")


def sign_test(a, b):
    """Teste do sinal exato (bilateral) nos pares em que só um dos dois acertou."""
    wins, losses = int((a & ~b).sum()), int((~a & b).sum())
    n = wins + losses
    p = min(1.0, 2 * sum(math.comb(n, k) for k in range(min(wins, losses) + 1)) / 2**n) if n else 1.0
    return wins, losses, p


t = acc.loc[triggered]
for label, a, b in [("H1", "rewind_kv", "blind_rewind"), ("H2", "rewind_kv", "baseline")]:
    wins, losses, p = sign_test(t[a], t[b])
    print(f"{label}: {a} vs {b}: {wins} vitórias, {losses} derrotas, p = {p:.3f}")

sub = df[df["problem_idx"].isin(triggered)].copy()
sub["n_rewinds"] = sub["rewinds"].apply(len)
print("\nSecundárias (problemas com detecção):")
display(sub.groupby("mode")[["correct", "generated_tokens", "hit_token_limit", "n_rewinds", "final_cache_len"]].mean())

### Problema a problema (só os com detecção)

In [ ]:
table = acc.loc[triggered].copy()
table["detecções"] = base.loc[triggered, "detections"]
display(table)